In [1]:
import os, rasterio, sys, pyflwdir, rioxarray
sys.path.append('backend/app/')
from rasterio.features import rasterize
from rasterio.mask import mask
from netCDF4 import Dataset, date2num
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
from shapely import force_2d
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
np.random.seed(42)

In [3]:
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, terrain):
    clipped = terrain.rio.clip(catchment.geometry, catchment.crs, drop=False)
    clipped = clipped.fillna(-9999)
    clipped.rio.write_nodata(-9999, inplace=True)
    return clipped

def write_tif(path, terrain, geo, col=''):
    transform = terrain.rio.transform()
    if col == '': shapes = ((geom, 1) for geom in geo.geometry)
    else: shapes = ((geom, value) for geom, value in zip(geo.geometry, geo[col]))
    raster = rasterize(
        shapes=shapes, out_shape=(terrain.rio.height, terrain.rio.width),
        transform=transform, fill=-9999, dtype="float32", all_touched=True
    )
    meta = {
        "driver": "GTiff", "height": terrain.rio.height,
        "width": terrain.rio.width, "count": 1, "dtype": "float32", 
        "crs": terrain.rio.crs, "transform": transform, "nodata": -9999
    }
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(raster, 1)

def create_forcing(out_path:str, terrain:rioxarray, time:pd.Series, values:np.ndarray, variable_name:str, unit:str, des:str):
    with Dataset(out_path, "w", format="NETCDF4") as nc:
        # Dimensions
        nc.createDimension("time", len(values))
        nc.createDimension("y", terrain.rio.height)
        nc.createDimension("x", terrain.rio.width)
        # Coordinates
        timestamp = nc.createVariable("time", "f8", ("time",))
        ys = nc.createVariable("y", "f4", ("y",))
        xs = nc.createVariable("x", "f4", ("x",))
        ys[:], xs[:] = terrain.y.values, terrain.x.values
        # Time metadata
        timestamp.units = "hours since 1970-01-01 00:00:00"
        timestamp.calendar = "standard"
        timestamp[:] = date2num(
            time.to_pydatetime(), units=timestamp.units, calendar=timestamp.calendar
        )
        var = nc.createVariable(
            variable_name, "f4", ("time", "y", "x"), zlib=True, 
            complevel=4, chunksizes=(1, 256, 256), fill_value=-9999
        )
        var.units = unit
        # Template array
        arr = np.ones( (terrain.rio.height, terrain.rio.width), dtype=np.float32)
        for i, val in tqdm(enumerate(values), total=len(values), desc=des):
            var[i,:,:] = arr * val


In [2]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
soil_path = os.path.join(sample_folder, 'soil.geojson')
land_path = os.path.join(sample_folder, 'land.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')

In [ ]:
# Write forcing
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']
forcings = {
    'precipitation': ['precip.nc', 'precip_mm', '(mm/h)'],
    'temperature':['temp.nc', 'temp_C', '(degC)'],
}
for key, value in forcings.items():
    path = f"{test_folder}/data/forcing/{value[0]}"
    create_forcing(
        path, terrain, weather_new.index, weather_new[value[1]].values, 
        key, value[2], f"Creating {key} forcing"
    )

In [13]:
# Clip dtm to catchment
catchment_UTM = catchment.to_crs(terrain.rio.crs)
terrain_clipped = clip_catchment(catchment_UTM, terrain)
terrain_out_path = os.path.normpath(os.path.join(f'{test_folder}/data/dem', "dtm_clipped.tif"))
terrain_clipped.rio.to_raster(terrain_out_path)

In [14]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.rio.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_layers = ['theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
for value in soil_layers:
    soil_path = os.path.normpath(os.path.join('test/data/soil', f'{value}.tif'))
    write_tif(soil_path, terrain, soil_UTM, value)

In [8]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.rio.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join('test/data/landuse', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [30]:
# Create raster and lookup table (used for calibration)
land_class = land.to_crs(terrain.rio.crs).copy()
land_class['class'] = None
columns, table = np.unique(land_class['land'].values), {}
for id, item in enumerate(columns):
    temp = land_class[land_class['land'] == item]
    table[item] = np.float32(temp.iloc[0][land_layers].values)
    land_class.loc[temp.index, 'class'] = id
land_class = land_class[['class', 'geometry']]
land_class_path = os.path.normpath(os.path.join('test/data/lookup', 'land_classes.tif'))
write_tif(land_class_path, terrain, land_class, 'class')
# Create lookup table
lookup = pd.DataFrame.from_dict(table, orient='index', columns=land_layers)
lookup.index.name = 'landcover'
lookup.reset_index(inplace=True)
lookup.insert(0, 'class_id', lookup.index)
# Save lookup table to csv
lookup_csv_path = os.path.normpath(os.path.join('test/data/lookup', 'lookup_land.csv'))
lookup.to_csv(lookup_csv_path, index=False)

In [75]:
# Process river
river_UTM = river.to_crs(terrain.rio.crs)
cols = {'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_UTM = river_UTM[river_UTM.is_valid].reset_index(drop=True)
river_UTM["geometry"] = river_UTM.geometry.apply(lambda g: force_2d(g))
river_dict = {
    'river': '', 'river_width': 'width', 'river_depth': 'depth', 'river_n': 'manning_n'
}
for key, value in river_dict.items():
    river_path = os.path.normpath(os.path.join(f'{test_folder}/data/river', f'{key}.tif'))
    write_tif(river_path, terrain, river_UTM, value)

In [57]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if not os.path.exists(model_path): os.makedirs(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "artifact_data" -v

2026-05-13 19:32:39,324 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-13 19:32:39,758 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Reading data catalog artifact_data latest
2026-05-13 19:32:39,758 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Users\vanln\.hydromt\artifact_data\v1.0.0\data_catalog.yml
2026-05-13 19:32:40,501 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-13 19:32:40,501 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-13 19:32:40,523 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-13 19:32:40,523 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-13 19:32:40,

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Envs\hyd_ai\Scripts\hydromt.exe\__main__.py", line 5, in <module>
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1514, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1435, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1902, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 1298, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Envs\hyd_ai\Lib\site-packages\click\core.py", line 853, in invoke
    return callback(*args, **kwargs)
        

In [18]:
with rasterio.open(terrain_path, "r") as src:
    elevtn = src.read(1).astype(np.float32)
    nodata = src.nodata
    if src.nodata is not None:
        elevtn[elevtn == src.nodata] = np.nan
    transform = src.transform
    crs = src.crs

In [20]:
flw = pyflwdir.from_dem(data=elevtn, nodata=np.nan, transform=src.transform, latlon=src.crs.is_geographic)
flw

In [ ]:
feat = flw.streams()
gdf = gpd.GeoDataFrame.from_features(feat, crs=crs)
gdf.to_file('streams.geojson', driver='GeoJSON')

In [24]:
gdf.to_file('streams.geojson', driver='GeoJSON')

In [ ]:

weather

,datetime,precip_mm,temp_C,shortwave_Wm2,longwave_Wm2,wind_mps,relhum_pct,pressure
0,2025-01-01 00:00:00,2.447,5.172,0.000,320.125,7.620,86.733,1009.946
1,2025-01-01 01:00:00,2.383,5.172,129.410,384.656,7.993,74.095,987.605
2,2025-01-01 02:00:00,3.192,5.172,250.000,384.239,8.049,70.175,997.114
3,2025-01-01 03:00:00,0.759,5.172,353.553,364.490,7.289,87.534,988.409
4,2025-01-01 04:00:00,1.788,5.172,433.013,322.710,8.959,91.852,1013.289
...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00,1.529,5.000,0.000,343.680,5.803,71.896,990.648
8756,2025-12-31 20:00:00,0.724,5.000,0.000,301.795,9.012,93.424,1017.078
8757,2025-12-31 21:00:00,0.415,5.000,0.000,378.202,9.810,85.269,1017.871
8758,2025-12-31 22:00:00,1.135,5.000,0.000,360.407,9.215,84.581,1012.472
